# Store Sales — per-horizon models

Every feature in the production model is shifted by **at least 16 days**, because the forecast
has to reach 16 days past the last observation. That is correct for 2017-08-31 and *wasteful*
for 2017-08-16, which is only one day out and could legitimately use yesterday's sales.

This notebook trains **four models instead of one**, each using the freshest data its horizon
range legally allows, and assembles the 16-day forecast from them:

| horizon | model | what it may see |
|---|---|---|
| 1 | `lag ≥ 1` | up to yesterday |
| 2 | `lag ≥ 2` | up to two days ago |
| 3–4 | `lag ≥ 4` | up to four days ago |
| 5–16 | `lag ≥ 16` | the production model, unchanged |

**The legality rule, stated once and asserted in code.** A target date `d` at horizon `h` has
forecast origin `o = d − h`. Feature `lag_k` is `sales(d − k)`, which is known at the origin
iff `d − k ≤ d − h`, i.e. **`k ≥ h`**. Getting this backwards would leak future data into the
holdout and produce a score that looks excellent locally and collapses on the leaderboard, so
the assembly is checked rather than trusted.

## What was measured

Paired across seeds 42/79/116 on the 2017-07-31 → 08-15 holdout, against the **same harness**
running the production configuration on all 16 days (never against an externally published
number — this project has a documented case of harness numbers drifting run to run):

| | |
|---|---|
| mean improvement | **−0.00238** |
| standard deviation | 0.00036 |
| seeds improving | **3 / 3** |
| significance | **6.6 σ** |

The gain concentrates where the mechanism predicts: horizons 1–4 improved by ~0.009, horizons
5–8 by ~0, and horizons 9–16 by **exactly 0.00000** — the last being a sanity check rather than
a result, since that bucket *is* the production model.

Five different bucket layouts were compared; all five beat the control on 3/3 seeds, and they
span only 0.0004 between them. The experiment establishes that bucketing works far more firmly
than it establishes which layout is best. This one had the best mean and beat the simplest
2-bucket alternative on 3/3 paired seeds (2.7 σ).

**Reference points.** Production holdout 0.39081, leaderboard 0.42074. The seed-noise floor on
this holdout is 0.0039 — but that ruler measures spread across seeds on the whole holdout,
while these arms share seeds *and* share 14,256 identical rows, which is why a paired test is
the right instrument and 6.6 σ is meaningful despite the absolute number being under 0.0039.

It also applies one post-processing rule: **series with no sales at all in the preceding 365
days are forced to exactly zero**. A pooled model cannot express exact zero and leaks small
positive forecasts onto structurally dead combinations; under RMSLE those rows are 3.6% of the
forecast, 0.01% of the units, and disproportionately expensive. Measured at **−0.00274
(3/3 seeds, 4.5 σ)** — see the section before the submission cell.

**Runtime ≈ 2.5 hours** (40 model fits). GPU is not needed.

In [1]:
import gc, time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

pd.set_option("display.width", 140)

COMP = "store-sales-time-series-forecasting"
REQUIRED = {"train.csv", "test.csv", "stores.csv", "holidays_events.csv"}

def has_data(p: Path) -> bool:
    try:
        return p.is_dir() and REQUIRED.issubset({f.name for f in p.iterdir() if f.is_file()})
    except OSError:
        return False

def find_data() -> Path:
    root = Path("/kaggle/input")
    if root.is_dir():
        for cand in [root / COMP, *sorted(d for d in root.iterdir() if d.is_dir())]:
            if has_data(cand):
                return cand
    for cand in (Path("data"), Path("../data"), Path("../../data")):
        if has_data(cand):
            return cand
    try:
        import kagglehub
        got = Path(kagglehub.competition_download(COMP))
        if has_data(got):
            return got
        for sub in got.rglob("*"):
            if has_data(sub):
                return sub
    except Exception as exc:
        print(f"kagglehub fallback failed: {exc}")
    raise FileNotFoundError(
        f"Could not find {sorted(REQUIRED)}. In the Kaggle editor: + Add Input -> "
        "Competitions -> store-sales-time-series-forecasting.")

DATA = find_data()
ON_KAGGLE = Path("/kaggle/working").is_dir()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("submissions/horizon")
OUT.mkdir(parents=True, exist_ok=True)

HORIZON = 16
MODEL_START = pd.Timestamp("2015-01-01")
EQ_START, EQ_END = pd.Timestamp("2016-04-16"), pd.Timestamp("2016-04-22")

# The validated production model's chain-wide 16-day total, used as a final sanity check.
# A blend bug once shipped half this volume and scored 0.66126 because nobody read the number.
VALIDATED_VOLUME = 12_932_032

DTYPES = {"store_nbr": "int8", "family": "category", "onpromotion": "int32", "sales": "float32"}
train  = pd.read_csv(DATA / "train.csv", parse_dates=["date"], dtype=DTYPES)
test   = pd.read_csv(DATA / "test.csv",  parse_dates=["date"], dtype=DTYPES)
stores = pd.read_csv(DATA / "stores.csv", dtype={"store_nbr": "int8"})
hol    = pd.read_csv(DATA / "holidays_events.csv", parse_dates=["date"])

TRAIN_END   = train.date.max()
VALID_START = TRAIN_END - pd.Timedelta(days=HORIZON - 1)
print(f"train {train.date.min():%Y-%m-%d} -> {TRAIN_END:%Y-%m-%d} | "
      f"test {test.date.min():%Y-%m-%d} -> {test.date.max():%Y-%m-%d}")

train 2013-01-01 -> 2017-08-15 | test 2017-08-16 -> 2017-08-31


---
## Panel, earthquake repair, holidays — unchanged from production

Christmas days restored as zero-sales (they are absent, not zero, in the raw file), and the April-2016 earthquake week replaced by each series' same-weekday median from the surrounding 8 weeks — the shock enters through lags and rolling windows for up to 135 days, so it is repaired at source rather than flagged.

In [2]:
t0 = time.time()
full_idx = pd.date_range(train.date.min(), test.date.max(), freq="D")

both = pd.concat([train.drop(columns="sales"), test], ignore_index=True)
both["family"] = both.family.astype(str)

sales_w = (train.assign(family=train.family.astype(str))
           .pivot(index="date", columns=["store_nbr", "family"], values="sales")
           .reindex(full_idx).sort_index(axis=1))
promo_w = (both.pivot(index="date", columns=["store_nbr", "family"], values="onpromotion")
           .reindex(full_idx).sort_index(axis=1).fillna(0.0))

xmas = pd.DatetimeIndex([d for d in full_idx
                         if d <= TRAIN_END and d not in set(train.date.unique())])
sales_w.loc[xmas] = 0.0
assert sales_w.loc[:TRAIN_END].isna().sum().sum() == 0

SERIES = sales_w.columns
n_s = len(SERIES)

def repair_window(W, start, end, halo_weeks=8):
    out = W.copy()
    win = pd.date_range(start, end)
    ctx = W.loc[start - pd.Timedelta(weeks=halo_weeks): end + pd.Timedelta(weeks=halo_weeks)]
    ctx = ctx.drop(index=win, errors="ignore")
    by_dow = ctx.groupby(ctx.index.dayofweek).median()
    for d in win:
        out.loc[d] = by_dow.loc[d.dayofweek].values
    return out

sales_w = repair_window(sales_w, EQ_START, EQ_END)
eq_dates = pd.date_range(EQ_START, EQ_END)
print(f"panel {sales_w.shape}, {n_s} series  ({time.time()-t0:.0f}s)")

panel (1704, 1782), 1782 series  (25s)


In [3]:
h = hol.copy()
h = h[~((h.type == "Holiday") & (h.transferred))]
h.loc[h.type == "Transfer", "type"] = "Holiday"
work_days = set(h.loc[h.type == "Work Day", "date"])
h = h[h.type != "Work Day"]
events = h[h.type == "Event"]
h = h[h.type != "Event"]

nat = pd.DatetimeIndex(sorted(set(h.loc[h.locale == "National", "date"])))
nat_name = (h[h.locale == "National"].drop_duplicates("date")[["date", "description"]]
            .rename(columns={"description": "nat_hol_name"}))
loc_name = (h[h.locale == "Local"].drop_duplicates(["date", "locale_name"])
            [["date", "locale_name", "description"]]
            .rename(columns={"locale_name": "city", "description": "loc_hol_name"}))

geo = stores.set_index("store_nbr")[["city", "state", "type", "cluster"]]
geo.columns = ["city", "state", "store_type", "cluster"]
assert not (set(loc_name.city) - set(stores.city))

pos = np.searchsorted(nat.values, full_idx.values)
prev_d = np.where(pos > 0, (full_idx.values - nat.values[np.maximum(pos - 1, 0)])
                  / np.timedelta64(1, "D"), 999)
next_d = np.where(pos < len(nat), (nat.values[np.minimum(pos, len(nat) - 1)] - full_idx.values)
                  / np.timedelta64(1, "D"), 999)
cal_hol = pd.DataFrame({"date": full_idx,
                        "work_day": full_idx.isin(work_days).astype("int8"),
                        "is_event": full_idx.isin(set(events.date)).astype("int8"),
                        "days_since_nat": np.clip(prev_d, 0, 30).astype("int16"),
                        "days_to_nat": np.clip(next_d, 0, 30).astype("int16")})
print(f"{len(nat_name)} national + {len(loc_name)} local holiday dates")

102 national + 147 local holiday dates


---
## Features, parameterised by minimum lag

The production feature set, with every backward-looking window shifted by `min_lag` instead of
a hard-coded 16. Promotion features are **exempt** — `onpromotion` is given for the whole test
window, so current-day and short-lead values are legal at any horizon and are identical across
all four models.

Two details worth stating because they are easy to get subtly wrong:

- **The lag ladder keeps its shape**, offset to the minimum: production's
  `16,17,…,22,28,35,49,63` is `min + (0,1,2,3,4,5,6,12,19,33,47)`, so the `lag ≥ 4` model uses
  `4,5,…,10,16,23,37,51`. Same structure, fresher data.
- **Same-weekday features start at the first multiple of 7 at or beyond `min_lag`** — 21 days
  for production, 7 for the `lag ≥ 1` and `lag ≥ 4` models. Anything else would mix weekdays
  and destroy the feature's meaning.

In [4]:
L = np.log1p(sales_w).astype("float32")
P = np.log1p(promo_w).astype("float32")
ZERO = (sales_w == 0).astype("float32")

# Promotion block -- identical for every model, built once.
promo_feats = {}
promo_feats["promo"] = P
promo_feats["promo_rmean_7"] = P.rolling(7, min_periods=1).mean()
promo_feats["promo_rmean_28"] = P.rolling(28, min_periods=3).mean()
promo_feats["promo_lag_16"] = P.shift(HORIZON)
promo_feats["promo_rel_112"] = P - P.rolling(112, min_periods=14).mean()
promo_feats["promo_rel_28"] = P - P.rolling(28, min_periods=5).mean()
for k in (1, 2, 3, 7):
    promo_feats[f"promo_lead_{k}"] = P.shift(-k)
promo_feats["promo_fwd7"] = P.shift(-6).rolling(7, min_periods=1).mean()

chain = P.mean(axis=1)
ones = np.ones(n_s, dtype="float32")
promo_feats["promo_chain_level"] = pd.DataFrame(
    np.outer(chain.to_numpy(dtype="float32"), ones), index=full_idx, columns=SERIES)
promo_feats["promo_chain_rel"] = pd.DataFrame(
    np.outer((chain - chain.rolling(112, min_periods=14).mean()).to_numpy(dtype="float32"),
             ones), index=full_idx, columns=SERIES)
del chain; gc.collect()

promo_raw = np.expm1(P)
fam_plog = np.log1p(promo_raw.T.groupby(level=1).sum().T)
store_plog = np.log1p(promo_raw.T.groupby(level=0).sum().T)
def _bcast(agg, level):
    return agg[SERIES.get_level_values(level)].set_axis(SERIES, axis=1)
promo_feats["fam_promo_rel"] = _bcast(fam_plog - fam_plog.rolling(112, min_periods=14).mean(), 1)
promo_feats["fam_promo_fwd7"] = _bcast(fam_plog.shift(-6).rolling(7, min_periods=1).mean()
                                       - fam_plog.rolling(112, min_periods=14).mean(), 1)
promo_feats["store_promo_rel"] = _bcast(
    store_plog - store_plog.rolling(112, min_periods=14).mean(), 0)
del promo_raw, fam_plog, store_plog; gc.collect()

# days_since_sale is a cumulative scan; compute the unshifted version once and shift per model.
A_pos = (sales_w.to_numpy() > 0)
gap = np.empty(A_pos.shape, dtype="float32")
_last = np.full(A_pos.shape[1], -999.0)
for i in range(A_pos.shape[0]):
    gap[i] = i - _last
    _last = np.where(A_pos[i], float(i), _last)
GAP = pd.DataFrame(np.minimum(gap, 999.0), index=sales_w.index, columns=SERIES)
del gap, A_pos; gc.collect()

mask = full_idx >= MODEL_START
dates_sel = full_idx[mask]
LAG_OFFSETS = (0, 1, 2, 3, 4, 5, 6, 12, 19, 33, 47)
print(f"{len(promo_feats)} promotion features built ({time.time()-t0:.0f}s)")

16 promotion features built (26s)


In [5]:
def build_design(min_lag):
    # Full design matrix with every backward-looking feature shifted by >= min_lag.
    feats = dict(promo_feats)
    for d in LAG_OFFSETS:
        feats[f"lag_{min_lag + d}"] = L.shift(min_lag + d)
    base = L.shift(min_lag)
    for w in (7, 14, 28, 56, 112):
        feats[f"rmean_{w}"] = base.rolling(w, min_periods=max(2, w // 4)).mean()
    for w in (14, 28):
        feats[f"rstd_{w}"] = base.rolling(w, min_periods=max(2, w // 4)).std()
    feats["rmax_28"] = base.rolling(28, min_periods=7).max()
    dow_start = 7 * int(np.ceil(min_lag / 7))     # first same-weekday lag at or beyond min_lag
    feats["dow_mean_4"] = sum(L.shift(dow_start + 7 * k) for k in range(4)) / 4
    feats["dow_mean_8"] = sum(L.shift(dow_start + 7 * k) for k in range(8)) / 8
    feats["zfrac_28"] = ZERO.shift(min_lag).rolling(28, min_periods=7).mean()
    feats["zfrac_112"] = ZERO.shift(min_lag).rolling(112, min_periods=28).mean()
    feats["days_since_sale"] = GAP.shift(min_lag)

    d = pd.DataFrame({
        "date": np.repeat(dates_sel.values, n_s),
        "store_nbr": np.tile(SERIES.get_level_values(0).to_numpy(), len(dates_sel)),
        "family": np.tile(SERIES.get_level_values(1).to_numpy(), len(dates_sel)),
    })
    for name, W in feats.items():
        d[name] = W.to_numpy(dtype="float32")[mask].ravel()
    d["target"] = L.to_numpy(dtype="float32")[mask].ravel()
    del feats; gc.collect()

    d = d.join(geo, on="store_nbr")
    dt = d.date
    d["dow"] = dt.dt.dayofweek.astype("int8")
    d["day"] = dt.dt.day.astype("int8")
    d["month"] = dt.dt.month.astype("int8")
    d["year"] = dt.dt.year.astype("int16")
    d["dayofyear"] = dt.dt.dayofyear.astype("int16")
    d["is_weekend"] = (d.dow >= 5).astype("int8")
    d["days_to_month_end"] = (dt.dt.days_in_month - dt.dt.day).astype("int8")
    d["payday_window"] = (dt.dt.day.isin([15, 16, 17, 1, 2, 3]) |
                          (d.days_to_month_end <= 1)).astype("int8")
    d = d.merge(cal_hol, on="date", how="left")
    d = d.merge(nat_name, on="date", how="left")
    d = d.merge(loc_name, on=["date", "city"], how="left")
    d["nat_hol_name"] = d.nat_hol_name.fillna("none")
    d["loc_hol_name"] = d.loc_hol_name.fillna("none")
    for c in ("family", "city", "state", "store_type", "nat_hol_name", "loc_hol_name"):
        d[c] = d[c].astype("category")
    d["store_nbr"] = d.store_nbr.astype("int16")
    d["cluster"] = d.cluster.astype("int16")
    cols = [c for c in d.columns if c not in ("date", "target")]
    assert len(cols) == 60, f"expected 60 features like production, got {len(cols)}"
    return d, cols

CATS = ["family", "city", "state", "store_type", "nat_hol_name", "loc_hol_name"]
PARAMS = dict(objective="regression", metric="rmse", learning_rate=0.08,
              num_leaves=96, min_data_in_leaf=50, feature_fraction=0.75,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              feature_pre_filter=False, num_threads=0, verbose=-1, seed=42)
SEED_CFG = [(42, 0.75, 0.80), (79, 0.60, 0.90), (116, 0.85, 0.70),
            (153, 0.70, 0.85), (190, 0.65, 0.75)]

def seed_params(i):
    p = dict(PARAMS)
    p["seed"], p["feature_fraction"], p["bagging_fraction"] = SEED_CFG[i]
    return p

def rmsle(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(np.asarray(y_true, float))) ** 2)))

# horizon -> minimum lag of the model serving it
ASSEMBLY = {1: 1, 2: 2, 3: 4, 4: 4, **{h: 16 for h in range(5, 17)}}
MIN_LAGS = sorted(set(ASSEMBLY.values()))

# Legality: origin of a date at horizon h is d-h; lag_k is sales(d-k), known iff k >= h.
for hh, ml in ASSEMBLY.items():
    assert ml >= hh, f"horizon {hh} served by lag>={ml} would read post-origin data"
assert set(ASSEMBLY) == set(range(1, 17))
print(f"assembly OK: {len(MIN_LAGS)} models {MIN_LAGS} covering 16 horizons")

assembly OK: 4 models [1, 2, 4, 16] covering 16 horizons


---
## Validation — holdout 2017-07-31 → 08-15

Each model early-stops **only on the horizons it will actually serve**. On any other day its features would be reading data published after the forecast origin, and stopping against those rows would be leakage. This is the fix for the earlier per-horizon attempt, which early-stopped each model against a single day (1,782 rows) and drew a false conclusion from it.

In [6]:
val_pred_log, best_rounds = {}, {}
for min_lag in MIN_LAGS:
    t = time.time()
    d, cols = build_design(min_lag)
    eq_row = d.date.isin(eq_dates)
    tr_m = (d.date < VALID_START) & d.target.notna() & ~eq_row
    va_m = (d.date >= VALID_START) & (d.date <= TRAIN_END)
    va_h = ((d.loc[va_m, "date"].to_numpy() - np.datetime64(VALID_START))
            / np.timedelta64(1, "D")).astype(int) + 1
    serves = np.isin(va_h, [hh for hh, ml in ASSEMBLY.items() if ml == min_lag])

    es_mask = np.zeros(len(d), dtype=bool)
    es_mask[np.flatnonzero(va_m.to_numpy())[np.flatnonzero(serves)]] = True

    dtr = lgb.Dataset(d.loc[tr_m, cols], d.loc[tr_m, "target"],
                      categorical_feature=CATS, free_raw_data=False)
    dva = lgb.Dataset(d.loc[es_mask, cols], d.loc[es_mask, "target"],
                      categorical_feature=CATS, reference=dtr, free_raw_data=False)
    m0 = lgb.train(seed_params(0), dtr, num_boost_round=2400, valid_sets=[dva],
                   callbacks=[lgb.early_stopping(150, verbose=False)])
    best_rounds[min_lag] = m0.num_trees()
    logs = [m0.predict(d.loc[va_m, cols])]
    for i in range(1, len(SEED_CFG)):
        mi = lgb.train(seed_params(i), dtr, num_boost_round=best_rounds[min_lag])
        logs.append(mi.predict(d.loc[va_m, cols]))
        del mi; gc.collect()
    val_pred_log[min_lag] = np.mean(logs, axis=0)

    if min_lag == 16:      # keep the pieces needed for scoring and for the control arm
        y_va = np.expm1(d.loc[va_m, "target"].to_numpy())
        y_va_log = d.loc[va_m, "target"].to_numpy()
        VA_H = va_h
    print(f"lag>={min_lag:>2}: {best_rounds[min_lag]:>5} trees, "
          f"early-stop rows {int(es_mask.sum()):>6,}  ({time.time()-t:.0f}s)", flush=True)
    del d, dtr, dva, m0, logs; gc.collect()

lag>= 1:   920 trees, early-stop rows  1,782  (476s)
lag>= 2:   508 trees, early-stop rows  1,782  (293s)
lag>= 4:  1813 trees, early-stop rows  3,564  (783s)
lag>=16:  1388 trees, early-stop rows 21,384  (649s)


In [7]:
def assemble(pred_by_lag):
    out = np.empty(len(VA_H))
    for hh, ml in ASSEMBLY.items():
        sel = VA_H == hh
        out[sel] = pred_by_lag[ml][sel]
    return out

control_log = val_pred_log[16]                 # production configuration on all 16 days
horizon_log = assemble(val_pred_log)

s_control = rmsle(y_va, np.expm1(control_log))
s_horizon = rmsle(y_va, np.expm1(horizon_log))
print(f"control  (production, lag>=16 everywhere) : {s_control:.5f}")
print(f"per-horizon assembly                      : {s_horizon:.5f}")
print(f"delta                                     : {s_horizon - s_control:+.5f}")
print()
rows = []
for ml in MIN_LAGS:
    hs = [hh for hh, m in ASSEMBLY.items() if m == ml]
    sel = np.isin(VA_H, hs)
    rows.append(dict(model=f"lag>={ml}", horizons=f"{min(hs)}-{max(hs)}", n=int(sel.sum()),
                     control=rmsle(y_va[sel], np.expm1(control_log[sel])),
                     horizon=rmsle(y_va[sel], np.expm1(horizon_log[sel]))))
bd = pd.DataFrame(rows)
bd["delta"] = bd.horizon - bd.control
print(bd.to_string(index=False, float_format=lambda v: f"{v: .5f}"))
print("\nThe lag>=16 row must show delta exactly 0.00000 -- it is the same model as the "
      "control on those rows. Any other value means the assembly is wired wrong.")
sanity = abs(bd.loc[bd.model == "lag>=16", "delta"].iloc[0])
assert sanity < 1e-12, f"assembly wiring error: lag>=16 delta is {sanity}, expected 0"


control  (production, lag>=16 everywhere) : 0.39117
per-horizon assembly                      : 0.38866
delta                                     : -0.00251

  model horizons     n  control  horizon    delta
 lag>=1      1-1  1782  0.38916  0.38007 -0.00909
 lag>=2      2-2  1782  0.38346  0.36759 -0.01588
 lag>=4      3-4  3564  0.37951  0.37128 -0.00823
lag>=16     5-16 21384  0.39388  0.39388  0.00000

The lag>=16 row must show delta exactly 0.00000 -- it is the same model as the control on those rows. Any other value means the assembly is wired wrong.


---
## Forcing structurally dead series to zero

A pooled model cannot express *exact* zero. It shares parameters across all 1,782 series, so a
combination that has not sold a single unit in over a year still receives a small positive
forecast — this model predicts a mean of 0.68 units, and as much as 26, on such rows.

Under RMSLE that is expensive out of all proportion to the volume involved. The metric weights
every row equally, so predicting 0.68 where the truth is 0 costs `log1p(0.68)² ≈ 0.27` — the
same penalty as missing a 5,000-unit series by the same *relative* amount. These rows are
**3.6% of the forecast and 0.01% of the units**.

**Measured on the holdout, paired across 3 seeds**, zeroing any series with no sales at all in
the preceding 365 days gains **−0.00274 (sd 0.00061, 3/3 seeds, 4.5 σ)** — larger than the
per-horizon change itself.

**The threshold is not tuned to an edge.** Sweeping it reveals a plateau:

| window | rows zeroed | rows that actually sold | Δ |
|---|---|---|---|
| 200–240 d | 1,136 | 28 | +0.00251 ❌ |
| 270–300 d | 1,120 | 21 | +0.00100 ❌ |
| **330–547 d** | **1,024–1,040** | **0** | **−0.00274** ✅ |
| 730 d | 912 | 0 | −0.00139 |
| 1000 d | 864 | 0 | −0.00083 |

365 sits mid-plateau with margin either side. The cliff below 330 days shows why the rule must
be conservative: adding 112 more rows introduces 28 that *did* sell, and that alone swings the
result by 0.005. Zeroing a row that sells 3 units costs `log1p(3)² = 1.92`, roughly seven times
what is saved by correctly zeroing a dead one — **the risk is deeply asymmetric, so the rule
buys nothing by being aggressive.** Combining it with prediction thresholds to catch more rows
was tested and added nothing measurable (−0.00277 at best, versus −0.00274).

This is CLAUDE.md's trap #7 finally being settled. The rule was tested years-of-sessions ago
on the *naive baseline*, where it changed the score by 0.00001 because recent-history methods
already predict ~0 on dead series. The note left behind read: *"it may still earn its keep on
a pooled model that leaks positive predictions — justify it on that model's score."* It does.

---
## Refit on all history and write the submission

In [8]:
te_pred_log = {}
for min_lag in MIN_LAGS:
    t = time.time()
    d, cols = build_design(min_lag)
    eq_row = d.date.isin(eq_dates)
    full_tr = (d.date <= TRAIN_END) & d.target.notna() & ~eq_row
    te_m = d.date > TRAIN_END
    dfull = lgb.Dataset(d.loc[full_tr, cols], d.loc[full_tr, "target"],
                        categorical_feature=CATS, free_raw_data=False)
    logs = []
    for i in range(len(SEED_CFG)):
        mi = lgb.train(seed_params(i), dfull, num_boost_round=best_rounds[min_lag])
        logs.append(mi.predict(d.loc[te_m, cols]))
        del mi; gc.collect()
    te_pred_log[min_lag] = np.mean(logs, axis=0)
    if min_lag == 16:
        te_key = d.loc[te_m, ["date", "store_nbr", "family"]].copy()
    print(f"lag>={min_lag:>2} refit on full history  ({time.time()-t:.0f}s)", flush=True)
    del d, dfull, logs; gc.collect()

TE_H = ((te_key.date.to_numpy() - np.datetime64(TRAIN_END)) / np.timedelta64(1, "D")).astype(int)
assert TE_H.min() == 1 and TE_H.max() == 16
final_log = np.empty(len(te_key))
for hh, ml in ASSEMBLY.items():
    sel = TE_H == hh
    final_log[sel] = te_pred_log[ml][sel]
pred = np.clip(np.expm1(final_log), 0, None)

lag>= 1 refit on full history  (445s)
lag>= 2 refit on full history  (285s)
lag>= 4 refit on full history  (778s)
lag>=16 refit on full history  (630s)


In [9]:
# Dormancy is judged on the RAW sales history, not the earthquake-repaired panel -- the repair
# only touches April 2016, which is outside this window anyway, but the raw file is the more
# defensible source for a "did this ever sell" question.
DORMANT_DAYS = 365
cutoff = TRAIN_END - pd.Timedelta(days=DORMANT_DAYS)
recent_total = (train[train.date > cutoff]
                .groupby(["store_nbr", "family"], observed=True).sales.sum())
dead = {(s, str(f)) for (s, f), v in recent_total.items() if v == 0}
# Series absent from the window entirely are dead too (a series that never traded at all).
all_keys = {(s, str(f)) for s, f in zip(test.store_nbr, test.family.astype(str))}
dead |= all_keys - {(s, str(f)) for s, f in recent_total.index}

te_keys = list(zip(te_key.store_nbr.to_numpy(), te_key.family.astype(str).to_numpy()))
dormant_mask = np.array([k in dead for k in te_keys])

leaked = pred[dormant_mask]
print(f"dormant series (no sales in {DORMANT_DAYS} days): {len(dead)} combinations, "
      f"{dormant_mask.sum():,} forecast rows ({dormant_mask.mean():.1%})")
print(f"the model was forecasting {leaked.sum():,.0f} units across them "
      f"(mean {leaked.mean():.3f}, max {leaked.max():.1f}) -- "
      f"{leaked.sum()/pred.sum():.3%} of total volume")
pred = pred.copy()
pred[dormant_mask] = 0.0

out = te_key.copy()
out["sales"] = pred
out["family"] = out.family.astype(str)

key = test.assign(family=test.family.astype(str))[["id", "date", "store_nbr", "family"]]
submission = key.merge(out, on=["date", "store_nbr", "family"], how="left")

assert len(submission) == len(test)
assert submission.sales.notna().all()
assert (submission.sales >= 0).all()
assert submission.id.equals(test.id)

# Volume check against the validated production submission. This is the cheap guard that would
# have caught an earlier broken submission (it shipped 6.4M units against 12.9M and scored
# 0.66126) in seconds, without needing to understand what had gone wrong.
volume = submission.sales.sum()
ratio = volume / VALIDATED_VOLUME
print(f"total predicted units : {volume:,.0f}")
print(f"validated reference   : {VALIDATED_VOLUME:,.0f}  (ratio {ratio:.3f})")
assert 0.85 < ratio < 1.15, f"volume is {ratio:.2f}x the validated model's -- do not submit"

daily = submission.groupby("date").sales.sum()
print("\ndaily chain-wide forecast volume:")
print(daily.to_string(float_format=lambda v: f"{v:,.0f}"))
print("\nCheck this for continuity before submitting -- a bucket-boundary discontinuity would "
      "show up as a step between day 2 and 3, or day 4 and 5, where the serving model changes.")
step = daily.pct_change().abs().max()
print(f"largest day-over-day change: {step:.1%}")

submission[["id", "sales"]].to_csv(OUT / "submission.csv", index=False)
print(f"\nwritten -> {(OUT / 'submission.csv').resolve()}")

dormant series (no sales in 365 days): 65 combinations, 1,040 forecast rows (3.6%)
the model was forecasting 725 units across them (mean 0.697, max 29.2) -- 0.006% of total volume
total predicted units : 12,714,454
validated reference   : 12,932,032  (ratio 0.983)

daily chain-wide forecast volume:
date
2017-08-16     826,952
2017-08-17     639,751
2017-08-18     764,361
2017-08-19     890,646
2017-08-20   1,036,356
2017-08-21     814,137
2017-08-22     732,943
2017-08-23     773,366
2017-08-24     646,120
2017-08-25     756,523
2017-08-26     909,507
2017-08-27   1,001,596
2017-08-28     762,121
2017-08-29     694,234
2017-08-30     771,880
2017-08-31     693,962

Check this for continuity before submitting -- a bucket-boundary discontinuity would show up as a step between day 2 and 3, or day 4 and 5, where the serving model changes.
largest day-over-day change: 23.9%

written -> /kaggle/working/submission.csv


---
## Reading the result

The honest expectation, and how to judge what comes back:

| Reference | Value |
|---|---|
| Production leaderboard | **0.42074** |
| Production holdout | 0.39081 |
| Measured local gain from this change | **−0.00238** (3/3 seeds, 6.6 σ) |

This project has a documented pattern in how changes transfer to the leaderboard, and it
depends on their **kind**. Changes that gave the model *information it did not previously have*
transferred at or above their measured value — the promotion rework measured **+0.0029
(worse)** locally and gained **−0.0802** on the leaderboard. Changes that merely let the same
model fit the same information better transferred at roughly 7% — the round cap measured
−0.00263 and gained −0.00019.

Per-horizon models are squarely in the first category: horizons 1–4 now see data that was
previously withheld from them. So the leaderboard gain could plausibly exceed −0.00238.

**But the honest downside case matters too.** Only 4 of 16 forecast days change at all, and
this project has had ten consecutive changes fail to transfer. A result anywhere between
−0.005 and +0.001 would be unsurprising. If it lands worse than 0.42074, the local measurement
was sound and the conclusion is about transfer, not about the experiment — record it and keep
the production model.